In [ ]:
# ── COLAB SETUP ────────────────────────────────────────────────────────────────
# Run this cell first. It embeds the required Gaussian output file (water.log)
# directly in this notebook so no external downloads are needed.
#
# No additional package installations are required — this notebook uses only
# Python standard library modules (math, os, etc.).

import sys, os, platform
print(f"Python {sys.version}")
print(f"Platform: {platform.system()}")

# Embedded content of water.log (B3LYP/6-31G* optimization + frequency of H2O)
_WATER_LOG_CONTENT = """
 Entering Gaussian System, Link 0=g16
 %chk=water.chk
 %mem=4GB
 %nprocshared=4
 ----------------------------------------------------------------------
 #p B3LYP/6-31G* Opt Freq

 Water molecule optimization and frequency calculation

 ----------------------------------------------------------------------
 Symbolic Z-matrix:
 Charge =  0 Multiplicity = 1
 O
 H  1  r1
 H  1  r1  2  a1

       Variables:
  r1                    0.96
  a1                  104.5


                          Input orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.000000
      2          1           0        0.000000    0.000000    0.960000
      3          1           0        0.927516    0.000000   -0.240000
 ---------------------------------------------------------------------

 SCF Done:  E(RB3LYP) =  -76.4083579211     A.U. after    9 cycles
            NFock= 9  Conv=0.47D-08     -V/T= 2.0089

 SCF Done:  E(RB3LYP) =  -76.4186251034     A.U. after   10 cycles
            NFock=10  Conv=0.38D-08     -V/T= 2.0091

 SCF Done:  E(RB3LYP) =  -76.4187385461     A.U. after    9 cycles
            NFock= 9  Conv=0.23D-08     -V/T= 2.0091

                         Standard orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.117176
      2          1           0        0.000000    0.757306   -0.468706
      3          1           0        0.000000   -0.757306   -0.468706
 ---------------------------------------------------------------------

 Rotational constants (GHZ):         836.3077963         434.9876987         286.8513882

 Zero-point correction=                           0.020908 (Hartree/Particle)
 Thermal correction to Energy=                    0.023742
 Thermal correction to Enthalpy=                  0.024686
 Thermal correction to Gibbs Free Energy=         0.003610
 Sum of electronic and zero-point Energies=     -76.397830
 Sum of electronic and thermal Energies=        -76.394996
 Sum of electronic and thermal Enthalpies=      -76.394052
 Sum of electronic and thermal Free Energies=  -76.415128

 E (Thermal)             CV                CP                S
 KCal/Mol        Cal/Mol-Kelvin    Cal/Mol-Kelvin    Cal/Mol-Kelvin
 14.896              6.002             8.314             44.278

 Zero-point vibrational energy     130943.2 (Joules/Mol)
                                    31.298 (Kcal/Mol)

 Frequencies --   1589.0765                3703.9127                3807.0960
 Red. masses --      1.0831                   1.0451                   1.0818
 Frc consts  --      1.6145                   8.4518                   9.2448
 IR Inten    --     67.1785                  10.4654                  49.3234

 Normal termination of Gaussian 16 at Mon Mar 14 12:34:56 2026.
"""

# Write to disk in the current working directory
if not os.path.exists("water.log"):
    with open("water.log", "w") as _f:
        _f.write(_WATER_LOG_CONTENT)
    print("Written : water.log")
else:
    print("Found existing: water.log")

print("Environment ready ✓")


# Python for Chemists — Part 2: Parsing Gaussian 16 Output Files

**Prerequisites:** This notebook assumes you have completed *Part 1: Core Concepts*.
You should be comfortable with variables, strings, lists, loops, dictionaries, functions,
and file I/O before continuing. Those tools are used here without re-explanation.

### What you will learn

| Section | Topic |
|---------|-------|
| 1 | Anatomy of a Gaussian `.log` file |
| 2 | Extracting SCF energies |
| 3 | Extracting thermochemical data |
| 4 | Verifying normal termination |
| 5 | Parsing the Standard orientation geometry |
| 6 | Computing bond lengths |
| 7 | Computing bond angles |
| 8 | A complete, reusable parser |

> **Reminder:** Replace each `None` stub with the correct value, delete
> `raise NotImplementedError()`, then run the **Test** cell to check your work.


---
## Section 1 — Anatomy of a Gaussian Output File

Gaussian writes a single plain-text `.log` (or `.out`) file recording everything that
happened during a calculation. The sections appear in this order for a geometry
optimisation followed by a frequency calculation:

```
Job header  (version, machine, route card)
Molecule specification  (charge, multiplicity, Z-matrix or Cartesian input)
━━━ Optimisation loop — repeats until convergence ━━━━━━━━━━━━━━━━━━━━━━━━━━
  Input orientation       ← atomic coordinates at this step
  SCF Done                ← electronic energy (Hartree) at this geometry
  Converged? → if not, compute gradient → take new step
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Standard orientation      ← final optimised coordinates in principal-axis frame
Rotational constants
━━━ Thermochemistry ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Zero-point correction
  Thermal corrections (Energy, Enthalpy, Gibbs)
  Sum of electronic + ZPE / H / G
  Harmonic vibrational frequencies
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Normal termination        ← ALWAYS verify this is present before using results
```

The cell below writes a realistic Gaussian output file for a water molecule
(B3LYP/6-31G* geometry optimisation + frequency calculation) that we will parse
throughout this notebook. **Run it now** so the file exists on disk.


In [1]:
# ── Create sample output file ─────────────────────────────────────────
gaussian_text = r"""
 Entering Gaussian System, Link 0=g16
 %chk=water.chk
 %mem=4GB
 %nprocshared=4
 ----------------------------------------------------------------------
 #p B3LYP/6-31G* Opt Freq

 Water molecule optimization and frequency calculation

 ----------------------------------------------------------------------
 Symbolic Z-matrix:
 Charge =  0 Multiplicity = 1
 O
 H  1  r1
 H  1  r1  2  a1

       Variables:
  r1                    0.96
  a1                  104.5


                          Input orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.000000
      2          1           0        0.000000    0.000000    0.960000
      3          1           0        0.927516    0.000000   -0.240000
 ---------------------------------------------------------------------

 SCF Done:  E(RB3LYP) =  -76.4083579211     A.U. after    9 cycles
            NFock= 9  Conv=0.47D-08     -V/T= 2.0089

 SCF Done:  E(RB3LYP) =  -76.4186251034     A.U. after   10 cycles
            NFock=10  Conv=0.38D-08     -V/T= 2.0091

 SCF Done:  E(RB3LYP) =  -76.4187385461     A.U. after    9 cycles
            NFock= 9  Conv=0.23D-08     -V/T= 2.0091

                         Standard orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.117176
      2          1           0        0.000000    0.757306   -0.468706
      3          1           0        0.000000   -0.757306   -0.468706
 ---------------------------------------------------------------------

 Rotational constants (GHZ):         836.3077963         434.9876987         286.8513882

 Zero-point correction=                           0.020908 (Hartree/Particle)
 Thermal correction to Energy=                    0.023742
 Thermal correction to Enthalpy=                  0.024686
 Thermal correction to Gibbs Free Energy=         0.003610
 Sum of electronic and zero-point Energies=     -76.397830
 Sum of electronic and thermal Energies=        -76.394996
 Sum of electronic and thermal Enthalpies=      -76.394052
 Sum of electronic and thermal Free Energies=  -76.415128

 E (Thermal)             CV                CP                S
 KCal/Mol        Cal/Mol-Kelvin    Cal/Mol-Kelvin    Cal/Mol-Kelvin
 14.896              6.002             8.314             44.278

 Zero-point vibrational energy     130943.2 (Joules/Mol)
                                    31.298 (Kcal/Mol)

 Frequencies --   1589.0765                3703.9127                3807.0960
 Red. masses --      1.0831                   1.0451                   1.0818
 Frc consts  --      1.6145                   8.4518                   9.2448
 IR Inten    --     67.1785                  10.4654                  49.3234

 Normal termination of Gaussian 16 at Mon Mar 14 12:34:56 2026.
"""

with open("water.log", "w") as fh:
    fh.write(gaussian_text)

with open("water.log", "r") as fh:
    all_lines = fh.readlines()

print(f"water.log written — {len(all_lines)} lines total")
print()
print("First 8 lines:")
for line in all_lines[:8]:
    print(repr(line))


water.log written — 75 lines total

First 8 lines:
'\n'
' Entering Gaussian System, Link 0=g16\n'
' %chk=water.chk\n'
' %mem=4GB\n'
' %nprocshared=4\n'
' ----------------------------------------------------------------------\n'
' #p B3LYP/6-31G* Opt Freq\n'
'\n'


Notice two things about how Python reads this file:

1. Each line is a **string** ending in `\n` — use `.strip()` to remove it.
2. Some lines have lots of leading whitespace — `.strip()` removes that too.

The string methods you practised in Part 1 (`strip`, `split`, `startswith`, `in`)
are the main tools for navigating this file.


---
## Section 2 — Extracting SCF Energies

The **SCF Done** line is printed once per optimisation step.
Each one looks like:

```
 SCF Done:  E(RB3LYP) =  -76.4187385461     A.U. after    9 cycles
```

After splitting on whitespace the tokens are:
```
index:  0         1          2   3   4                5       6  7  8
token: 'SCF'   'Done:'  'E(RB3LYP)' '=' '-76.4187385461' 'A.U.' ...
```

The energy is **always at index 4**.
For a geometry optimisation you want the **last** `SCF Done` line — the converged energy.


In [ ]:
# ── Example 2 — collect all SCF energies ────────────────────────────
H2K = 627.509   # Hartree → kcal/mol

scf_energies = []
with open("water.log", "r") as fh:
    for line in fh:
        if "SCF Done:" in line:
            energy = float(line.split()[4])
            scf_energies.append(energy)

print(f"SCF energies found: {len(scf_energies)}")
for i, e in enumerate(scf_energies, start=1):
    print(f"  Step {i}: {e:.10f} Hartree")

print()
print(f"Final (converged) energy : {scf_energies[-1]:.10f} Hartree")
print(f"Energy lowering (1→last) : {(scf_energies[-1] - scf_energies[0]) * H2K:.4f} kcal/mol")


#### ✏️ Practice 2
Using the `scf_energies` list from the example above, fill in the three result variables.

- `final_scf_energy`: the converged SCF energy (last value in the list), in Hartree
- `lowest_scf_energy`: the minimum SCF energy found across all steps
- `energy_lowering_kcal`: total energy change from step 1 to the last step, in kcal/mol
  (should be negative — the molecule got more stable)


In [ ]:
# ── Practice 2 ───────────────────────────────────────────────────────
# scf_energies and H2K are available from the example above

final_scf_energy      = None   # float (Hartree) — last SCF energy
lowest_scf_energy     = None   # float (Hartree) — minimum across all steps
energy_lowering_kcal  = None   # float (kcal/mol) — step 1 → last step

# YOUR CODE HERE
raise NotImplementedError()

print(f"Final SCF energy:    {final_scf_energy:.10f} Hartree")
print(f"Lowest SCF energy:   {lowest_scf_energy:.10f} Hartree")
print(f"Energy lowering:     {energy_lowering_kcal:.4f} kcal/mol")


In [ ]:
# ── Tests for Practice 2 ─────────────────────────────────────────────
assert abs(final_scf_energy - (-76.4187385461)) < 1e-8,     f"final_scf_energy should be -76.4187385461, got {final_scf_energy}"

assert abs(lowest_scf_energy - min(scf_energies)) < 1e-10,     f"lowest_scf_energy should be {min(scf_energies):.10f}, got {lowest_scf_energy:.10f}"

expected_lowering = (scf_energies[-1] - scf_energies[0]) * 627.509
assert abs(energy_lowering_kcal - expected_lowering) < 1e-4,     f"energy_lowering_kcal should be ≈ {expected_lowering:.4f}, got {energy_lowering_kcal:.4f}"
assert energy_lowering_kcal < 0,     "energy_lowering_kcal should be negative (energy decreased during optimisation)"

print("✓ All Practice 2 checks passed!")


---
## Section 3 — Thermochemical Corrections

After a frequency calculation Gaussian prints a thermochemistry block.
Each quantity appears on a line that **starts with a recognisable string**,
and the numerical value is always the **last token** on the line.

```
 Zero-point correction=                           0.020908 (Hartree/Particle)
 Thermal correction to Enthalpy=                  0.024686
 Thermal correction to Gibbs Free Energy=         0.003610
 Sum of electronic and zero-point Energies=     -76.397830
 Sum of electronic and thermal Enthalpies=      -76.394052
 Sum of electronic and thermal Free Energies=  -76.415128
```

The strategy: keep a dictionary mapping each search string to a short key name,
then scan the file once and populate the dictionary.


In [ ]:
# ── Example 3 — extract thermochemistry ─────────────────────────────
THERMO_KEYS = {
    "Zero-point correction=":                       "zpe_correction",
    "Thermal correction to Enthalpy=":              "thermal_H",
    "Thermal correction to Gibbs Free Energy=":     "thermal_G",
    "Sum of electronic and zero-point Energies=":   "E_zpe",
    "Sum of electronic and thermal Enthalpies=":    "H_total",
    "Sum of electronic and thermal Free Energies=": "G_total",
}

thermo = {}
with open("water.log", "r") as fh:
    for line in fh:
        stripped = line.strip()
        for search, key in THERMO_KEYS.items():
            if stripped.startswith(search):
                thermo[key] = float(stripped.split()[-1])

H2K = 627.509
print(f"{'Quantity':<28} {'Hartree':>14}  {'kcal/mol':>12}")
print("-" * 58)
for key, val in thermo.items():
    print(f"{key:<28} {val:>14.6f}  {val * H2K:>12.3f}")


#### ✏️ Practice 3
Using the `thermo` dictionary from the example above, fill in the result variables.

- `zpe_kcal`: zero-point energy correction in kcal/mol
- `delta_HG_kcal`: the entropic penalty — difference between total enthalpy and total
  Gibbs free energy, in kcal/mol (`G_total − H_total`)
  This is $-T\Delta S$ at 298 K and tells you how much entropy costs energetically.
- `pct_zpe_of_scf`: zero-point-corrected energy (`E_zpe`) expressed as a **percentage**
  of the raw SCF energy — i.e., how much the ZPE correction changes the energy relative
  to the SCF value: `(E_zpe − final_scf_energy) / abs(final_scf_energy) * 100`


In [ ]:
# ── Practice 3 ───────────────────────────────────────────────────────
# thermo dict, H2K, and final_scf_energy are available from earlier cells

zpe_kcal        = None   # float (kcal/mol) — zero-point correction in kcal/mol
delta_HG_kcal   = None   # float (kcal/mol) — G_total − H_total
pct_zpe_of_scf  = None   # float (%) — how much ZPE shifts the energy relative to SCF

# YOUR CODE HERE
raise NotImplementedError()

print(f"Zero-point energy:          {zpe_kcal:.3f} kcal/mol")
print(f"Entropic penalty (G − H):   {delta_HG_kcal:.3f} kcal/mol")
print(f"ZPE as % of SCF energy:     {pct_zpe_of_scf:.4f}%")


In [ ]:
# ── Tests for Practice 3 ─────────────────────────────────────────────
expected_zpe = thermo["zpe_correction"] * 627.509
assert abs(zpe_kcal - expected_zpe) < 1e-3,     f"zpe_kcal should be ≈ {expected_zpe:.3f}, got {zpe_kcal:.3f}"

expected_dHG = (thermo["G_total"] - thermo["H_total"]) * 627.509
assert abs(delta_HG_kcal - expected_dHG) < 1e-3,     f"delta_HG_kcal should be ≈ {expected_dHG:.3f}, got {delta_HG_kcal:.3f}"
assert delta_HG_kcal < 0,     "delta_HG_kcal (G - H = -TS) should be negative at 298 K"

expected_pct = (thermo["E_zpe"] - final_scf_energy) / abs(final_scf_energy) * 100
assert abs(pct_zpe_of_scf - expected_pct) < 1e-4,     f"pct_zpe_of_scf should be ≈ {expected_pct:.4f}%, got {pct_zpe_of_scf:.4f}%"

print("✓ All Practice 3 checks passed!")


---
## Section 4 — Verifying Normal Termination

> **Rule:** Never use results from a job that did not terminate normally.

If Gaussian crashes, runs out of time, or hits a disk quota, the last line will
be something like `Error termination` instead of `Normal termination`.
Always check before reporting numbers.


In [ ]:
# ── Example 4 — termination check ───────────────────────────────────
def check_termination(filename):
    """
    Return (terminated_normally, timestamp) for a Gaussian log file.
    timestamp is a string like 'Mon Mar 14 12:34:56 2026', or None.
    """
    with open(filename, "r") as fh:
        lines = fh.readlines()

    for line in lines:
        if "Normal termination" in line:
            tokens = line.split()
            idx = tokens.index("at")
            timestamp = " ".join(tokens[idx + 1:]).rstrip(".")
            return True, timestamp

    return False, None

ok, ts = check_termination("water.log")
if ok:
    print(f"✓ Job completed normally ({ts})")
else:
    print("✗ WARNING: job did NOT terminate normally — do not use these results!")


#### ✏️ Practice 4
Write a function `summarise_termination(filename)` that returns a **dictionary** with:

- `"normal"`: `True` or `False`
- `"timestamp"`: the date/time string, or `None`
- `"n_scf_steps"`: the number of `SCF Done` lines (i.e., optimisation steps)
- `"last_scf_energy"`: the final SCF energy as a float, or `None` if none found

The function name and its return variable name are pre-written below.


In [ ]:
# ── Practice 4 ───────────────────────────────────────────────────────
def summarise_termination(filename):
    """Return a summary dict for a Gaussian log file."""
    # YOUR CODE HERE
    raise NotImplementedError()

summary = summarise_termination("water.log")
for key, val in summary.items():
    print(f"  {key:<20}: {val}")


In [ ]:
# ── Tests for Practice 4 ─────────────────────────────────────────────
assert callable(summarise_termination), "summarise_termination should be a function"

s = summarise_termination("water.log")
assert isinstance(s, dict), "summarise_termination should return a dict"
assert s["normal"] is True, "normal should be True for water.log"
assert s["timestamp"] is not None, "timestamp should not be None"
assert "2026" in s["timestamp"], f"timestamp should contain the year, got {s['timestamp']!r}"
assert s["n_scf_steps"] == 3,     f"n_scf_steps should be 3 (three SCF Done lines), got {s['n_scf_steps']}"
assert abs(s["last_scf_energy"] - (-76.4187385461)) < 1e-8,     f"last_scf_energy should be -76.4187385461, got {s['last_scf_energy']}"

print("✓ All Practice 4 checks passed!")


---
## Section 5 — Parsing the Standard Orientation

The **Standard orientation** block contains the final atomic coordinates in Ångströms,
in the molecule's principal-axis frame. It appears after the last optimisation step:

```
                         Standard orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.117176
      2          1           0        0.000000    0.757306   -0.468706
      3          1           0        0.000000   -0.757306   -0.468706
 ---------------------------------------------------------------------
```

**Parsing strategy:** use a boolean flag that turns on when you see
`"Standard orientation:"` and turns off when you hit the closing dashes.
Reset the atoms list each time you enter a new block so you always keep the **last** one
(relevant for multi-step jobs).

Column layout of each coordinate line (after splitting on whitespace):

| Index | Content |
|-------|---------|
| 0 | Center number |
| 1 | **Atomic number** |
| 2 | Atomic type |
| 3 | **X** |
| 4 | **Y** |
| 5 | **Z** |


In [ ]:
# ── Example 5 — parse Standard orientation ──────────────────────────
ATOMIC_SYMBOLS = {
    1: "H",  5: "B",  6: "C",  7: "N",  8: "O",  9: "F",
   14: "Si", 15: "P", 16: "S", 17: "Cl", 35: "Br", 53: "I",
}

def parse_standard_orientation(filename):
    """
    Return a list of atom dicts from the last Standard orientation block.
    Each dict: {'symbol': str, 'x': float, 'y': float, 'z': float}
    """
    atoms        = []
    in_block     = False
    header_skip  = 0

    with open(filename, "r") as fh:
        for line in fh:
            if "Standard orientation:" in line:
                in_block, header_skip, atoms = True, 4, []
                continue
            if in_block:
                if header_skip > 0:
                    header_skip -= 1
                    continue
                if line.strip().startswith("-----"):
                    in_block = False
                    continue
                t = line.split()
                sym = ATOMIC_SYMBOLS.get(int(t[1]), f"Z{t[1]}")
                atoms.append({"symbol": sym,
                              "x": float(t[3]), "y": float(t[4]), "z": float(t[5])})
    return atoms

atoms = parse_standard_orientation("water.log")

print(f"{'#':<4} {'Sym':<6} {'X (Å)':>10} {'Y (Å)':>10} {'Z (Å)':>10}")
print("-" * 44)
for i, a in enumerate(atoms, 1):
    print(f"{i:<4} {a['symbol']:<6} {a['x']:>10.6f} {a['y']:>10.6f} {a['z']:>10.6f}")


#### ✏️ Practice 5
Using the `atoms` list from the example above, fill in the result variables.

- `n_atoms`: total number of atoms
- `n_oxygen`: number of oxygen atoms (symbol `"O"`)
- `n_hydrogen`: number of hydrogen atoms (symbol `"H"`)
- `formula`: molecular formula built from the counts, e.g. `"H2O"`
  (assume only H and O are present; put O first, then H)


In [ ]:
# ── Practice 5 ───────────────────────────────────────────────────────
# atoms list is available from the example above

n_atoms    = None   # int — total atoms
n_oxygen   = None   # int — number of O atoms
n_hydrogen = None   # int — number of H atoms
formula    = None   # str — e.g. "H2O"

# YOUR CODE HERE
raise NotImplementedError()

print(f"Total atoms: {n_atoms}")
print(f"Oxygen:      {n_oxygen}")
print(f"Hydrogen:    {n_hydrogen}")
print(f"Formula:     {formula}")


In [ ]:
# ── Tests for Practice 5 ─────────────────────────────────────────────
assert n_atoms == 3,    f"n_atoms should be 3, got {n_atoms}"
assert n_oxygen == 1,   f"n_oxygen should be 1, got {n_oxygen}"
assert n_hydrogen == 2, f"n_hydrogen should be 2, got {n_hydrogen}"
assert "O" in formula and "H" in formula,     f"formula should contain both O and H, got {formula!r}"
assert "2" in formula,  f"formula should show 2 hydrogens (e.g. H2), got {formula!r}"

print("✓ All Practice 5 checks passed!")


---
## Section 6 — Computing Bond Lengths

Given two atoms at positions $(x_1, y_1, z_1)$ and $(x_2, y_2, z_2)$,
the Euclidean distance is:

$$d = \sqrt{(x_2-x_1)^2 + (y_2-y_1)^2 + (z_2-z_1)^2}$$

This is implemented with Python's `math.sqrt` from the standard library,
which you import once at the top of the cell.


In [ ]:
# ── Example 6 — bond length ──────────────────────────────────────────
import math

def bond_length(atom1, atom2):
    """Euclidean distance (Å) between two atom dicts with keys x, y, z."""
    dx = atom2["x"] - atom1["x"]
    dy = atom2["y"] - atom1["y"]
    dz = atom2["z"] - atom1["z"]
    return math.sqrt(dx**2 + dy**2 + dz**2)

# atoms[0]=O, atoms[1]=H1, atoms[2]=H2
r_OH1 = bond_length(atoms[0], atoms[1])
r_OH2 = bond_length(atoms[0], atoms[2])
r_HH  = bond_length(atoms[1], atoms[2])

print(f"O–H1 : {r_OH1:.4f} Å")
print(f"O–H2 : {r_OH2:.4f} Å")
print(f"H–H  : {r_HH:.4f} Å")
print(f"Mean O–H : {(r_OH1 + r_OH2) / 2:.4f} Å  (experiment: 0.9572 Å)")


#### ✏️ Practice 6
Write a function `all_pairwise_distances(atoms)` that returns a **list of tuples**
`(label, distance)` for every unique atom pair, where `label` is a string like `"O1-H2"`.
Then use it to:

- Store the complete list as `pairs`
- Store the shortest distance as `shortest_dist`
- Store the label of the shortest pair as `shortest_label`


In [ ]:
# ── Practice 6 ───────────────────────────────────────────────────────
def all_pairwise_distances(atoms):
    """Return [(label, distance_Å), ...] for all unique atom pairs."""
    # YOUR CODE HERE
    raise NotImplementedError()

pairs          = None   # list of (label, distance) tuples
shortest_dist  = None   # float (Å) — smallest distance
shortest_label = None   # str — label of the shortest pair

# YOUR CODE HERE
raise NotImplementedError()

print(f"{'Pair':<10} {'Distance (Å)':>14}")
print("-" * 26)
for label, dist in sorted(pairs, key=lambda p: p[1]):
    print(f"{label:<10} {dist:>14.4f}")
print(f"
Shortest: {shortest_label} = {shortest_dist:.4f} Å")


In [ ]:
# ── Tests for Practice 6 ─────────────────────────────────────────────
assert callable(all_pairwise_distances), "all_pairwise_distances should be a function"
assert len(pairs) == 3, f"3 atoms → 3 unique pairs, got {len(pairs)}"

# All distances should be positive floats
for label, dist in pairs:
    assert isinstance(dist, float) and dist > 0,         f"Each distance should be a positive float, got {dist!r} for {label}"

# O-H bonds should be ~0.96 Å, H-H should be longer
oh_dists = [d for lbl, d in pairs if "O" in lbl and "H" in lbl]
assert len(oh_dists) == 2, "Should be 2 O-H pairs"
for d in oh_dists:
    assert 0.90 < d < 1.10, f"O-H distance should be ~0.96 Å, got {d:.4f}"

assert abs(shortest_dist - min(d for _, d in pairs)) < 1e-8
assert shortest_label == min(pairs, key=lambda p: p[1])[0]

print("✓ All Practice 6 checks passed!")


---
## Section 7 — Computing Bond Angles

The bond angle at atom **B** in the arrangement **A–B–C** is found using the dot product
of vectors $\vec{BA}$ and $\vec{BC}$:

$$\cos\theta = \frac{\vec{BA} \cdot \vec{BC}}{|\vec{BA}|\,|\vec{BC}|}
\qquad
\theta = \arccos\!\left(\frac{\vec{BA} \cdot \vec{BC}}{|\vec{BA}|\,|\vec{BC}|}\right)$$

Note the `max(-1, min(1, ...))` clamp in the example — floating-point rounding can
occasionally push the cosine just outside $[-1, 1]$, causing `math.acos` to raise a
`ValueError`.


In [ ]:
# ── Example 7 — bond angle ───────────────────────────────────────────
def bond_angle(atomA, atomB, atomC):
    """
    A–B–C angle in degrees. B is the central atom.
    Uses the dot-product formula with a clamp to avoid acos domain errors.
    """
    BA = [atomA[k] - atomB[k] for k in ("x", "y", "z")]
    BC = [atomC[k] - atomB[k] for k in ("x", "y", "z")]

    dot     = sum(BA[i] * BC[i] for i in range(3))
    mag_BA  = math.sqrt(sum(v**2 for v in BA))
    mag_BC  = math.sqrt(sum(v**2 for v in BC))
    cos_ang = max(-1.0, min(1.0, dot / (mag_BA * mag_BC)))
    return math.degrees(math.acos(cos_ang))

# H1–O–H2  (O is the central atom, index 0)
angle_HOH = bond_angle(atoms[1], atoms[0], atoms[2])
print(f"H–O–H angle:  {angle_HOH:.4f}°")
print(f"Experimental: 104.45°")
print(f"Difference:   {abs(angle_HOH - 104.45):.4f}°")


#### ✏️ Practice 7
Fill in the three result variables by applying `bond_angle` to `water.log`'s atoms.

- `angle_HOH`: the H–O–H angle in degrees (H1 is `atoms[1]`, O is `atoms[0]`, H2 is `atoms[2]`)
- `within_one_degree`: `True` if the calculated angle is within 1° of the experimental value (104.45°)
- `error_deg`: the absolute difference from experiment in degrees


In [ ]:
# ── Practice 7 ───────────────────────────────────────────────────────
experimental_HOH = 104.45   # degrees

angle_HOH        = None   # float (°) — computed H–O–H bond angle
within_one_degree = None  # bool — is |computed − experimental| < 1°?
error_deg        = None   # float (°) — absolute deviation from experiment

# YOUR CODE HERE
raise NotImplementedError()

print(f"Computed H–O–H:  {angle_HOH:.4f}°")
print(f"Experimental:    {experimental_HOH}°")
print(f"Error:           {error_deg:.4f}°")
print(f"Within 1°?       {within_one_degree}")


In [ ]:
# ── Tests for Practice 7 ─────────────────────────────────────────────
expected_angle = bond_angle(atoms[1], atoms[0], atoms[2])
assert abs(angle_HOH - expected_angle) < 1e-6,     f"angle_HOH should be ≈ {expected_angle:.4f}°, got {angle_HOH:.4f}°"
assert 100 < angle_HOH < 110,     f"H-O-H angle should be between 100° and 110°, got {angle_HOH:.2f}°"

expected_err = abs(expected_angle - 104.45)
assert abs(error_deg - expected_err) < 1e-6,     f"error_deg should be ≈ {expected_err:.4f}°, got {error_deg:.4f}°"

assert isinstance(within_one_degree, bool),     "within_one_degree should be a bool (True or False)"
assert within_one_degree == (expected_err < 1.0)

print("✓ All Practice 7 checks passed!")


---
## Section 8 — A Complete, Reusable Parser

Now we combine everything into a single function. Good design means:

- **One pass** through the file (open once, collect everything)
- A **clean return value** — a dictionary so callers can access results by name
- **Defensive defaults** — set everything to `None` or `[]` before the loop,
  so partial files don't crash the caller

Study the structure carefully: the same flag-based pattern for the Standard orientation
block and the same `startswith` pattern for thermochemistry — just combined.


In [ ]:
# ── Example 8 — complete Gaussian parser ────────────────────────────
import math

ATOMIC_SYMBOLS = {
    1: "H",  5: "B",  6: "C",  7: "N",  8: "O",  9: "F",
   14: "Si", 15: "P", 16: "S", 17: "Cl", 35: "Br", 53: "I",
}
THERMO_KEYS = {
    "Zero-point correction=":                       "zpe_correction",
    "Thermal correction to Enthalpy=":              "thermal_H",
    "Thermal correction to Gibbs Free Energy=":     "thermal_G",
    "Sum of electronic and zero-point Energies=":   "E_zpe",
    "Sum of electronic and thermal Enthalpies=":    "H_total",
    "Sum of electronic and thermal Free Energies=": "G_total",
}

def parse_gaussian(filename):
    """
    Parse a Gaussian log file in one pass.

    Returns
    -------
    dict with keys:
      normal_termination  (bool)
      timestamp           (str or None)
      scf_energies        (list of float)
      final_scf_energy    (float or None)
      thermo              (dict)
      atoms               (list of dicts with symbol, x, y, z)
    """
    result = {
        "normal_termination": False,
        "timestamp":          None,
        "scf_energies":       [],
        "final_scf_energy":   None,
        "thermo":             {},
        "atoms":              [],
    }
    in_std = False
    hskip  = 0

    with open(filename, "r") as fh:
        for line in fh:
            # 1. Normal termination
            if "Normal termination" in line:
                result["normal_termination"] = True
                tokens = line.split()
                idx = tokens.index("at")
                result["timestamp"] = " ".join(tokens[idx+1:]).rstrip(".")

            # 2. SCF energy
            if "SCF Done:" in line:
                result["scf_energies"].append(float(line.split()[4]))

            # 3. Thermochemistry
            stripped = line.strip()
            for search, key in THERMO_KEYS.items():
                if stripped.startswith(search):
                    result["thermo"][key] = float(stripped.split()[-1])

            # 4. Standard orientation
            if "Standard orientation:" in line:
                in_std, hskip, result["atoms"] = True, 4, []
                continue
            if in_std:
                if hskip > 0:
                    hskip -= 1; continue
                if line.strip().startswith("-----"):
                    in_std = False; continue
                t = line.split()
                sym = ATOMIC_SYMBOLS.get(int(t[1]), f"Z{t[1]}")
                result["atoms"].append({"symbol": sym,
                    "x": float(t[3]), "y": float(t[4]), "z": float(t[5])})

    if result["scf_energies"]:
        result["final_scf_energy"] = result["scf_energies"][-1]

    return result

# ─── Run the parser and show a summary ────────────────────────────────
data = parse_gaussian("water.log")
H2K  = 627.509

print("━" * 55)
print("  Gaussian Summary — water.log (B3LYP/6-31G*)")
print("━" * 55)
print(f"  Normal termination : {data['normal_termination']}  ({data['timestamp']})")
print(f"  SCF steps          : {len(data['scf_energies'])}")
print(f"  Final SCF energy   : {data['final_scf_energy']:.10f} Hartree")
print()
print("  Thermochemistry:")
for k, v in data["thermo"].items():
    print(f"    {k:<36}: {v:.6f} Hartree  ({v*H2K:.2f} kcal/mol)")
print()
print("  Geometry:")
for i, a in enumerate(data["atoms"], 1):
    print(f"    {i}  {a['symbol']}  {a['x']:>10.6f}  {a['y']:>10.6f}  {a['z']:>10.6f}")
print("━" * 55)


#### ✏️ Practice 8 — Extend the Parser

Add **vibrational frequency** support to `parse_gaussian`. A frequency line looks like:

```
 Frequencies --   1589.0765                3703.9127                3807.0960
```

After splitting, the numerical values start at index 2.

Return the extended result dict from a new function `parse_gaussian_v2(filename)`.
The result should include two new keys on top of everything in `parse_gaussian`:

- `"frequencies"`: a flat list of all vibrational frequencies in cm⁻¹ (floats)
- `"n_imaginary"`: count of negative frequencies (int) — should be 0 for a true minimum,
  1 for a transition state


In [ ]:
# ── Practice 8 ───────────────────────────────────────────────────────
def parse_gaussian_v2(filename):
    """Extended parser that also collects vibrational frequencies."""
    # YOUR CODE HERE
    raise NotImplementedError()

data2 = parse_gaussian_v2("water.log")

print(f"Frequencies (cm⁻¹): {data2['frequencies']}")
print(f"Imaginary freqs:    {data2['n_imaginary']}")
print()
if data2["n_imaginary"] == 0:
    print("✓ No imaginary frequencies — this is a true minimum on the potential energy surface.")
elif data2["n_imaginary"] == 1:
    print("⚠ One imaginary frequency — this is a transition state.")
else:
    print(f"✗ {data2['n_imaginary']} imaginary frequencies — geometry may not be converged.")


In [ ]:
# ── Tests for Practice 8 ─────────────────────────────────────────────
assert callable(parse_gaussian_v2), "parse_gaussian_v2 should be a function"

d = parse_gaussian_v2("water.log")
assert isinstance(d, dict), "parse_gaussian_v2 should return a dict"

# Must still have all keys from the base parser
for key in ("normal_termination", "scf_energies", "final_scf_energy", "thermo", "atoms"):
    assert key in d, f"Result dict is missing base key: '{key}'"

# Frequency keys
assert "frequencies" in d, "Result dict should have key 'frequencies'"
assert "n_imaginary" in d, "Result dict should have key 'n_imaginary'"

# Water has 3 atoms → 3N-6 = 3 vibrational modes
assert len(d["frequencies"]) == 3,     f"H₂O has 3 normal modes, got {len(d['frequencies'])}"

# All frequencies should be positive for a converged minimum
for f in d["frequencies"]:
    assert f > 0, f"All frequencies should be positive for a minimum, got {f:.2f} cm⁻¹"

assert d["n_imaginary"] == 0,     f"n_imaginary should be 0 for a converged minimum, got {d['n_imaginary']}"

# Check specific frequency values (from the log file)
expected_freqs = sorted([1589.0765, 3703.9127, 3807.0960])
got_freqs      = sorted(d["frequencies"])
for exp, got in zip(expected_freqs, got_freqs):
    assert abs(got - exp) < 0.001, f"Expected frequency {exp}, got {got}"

print("✓ All Practice 8 checks passed!")


---
## Summary

You now have a complete toolkit for reading Gaussian output files.

| Task | Key technique |
|------|--------------|
| Find SCF energies | `"SCF Done:" in line` → `line.split()[4]` |
| Extract thermochemistry | `stripped.startswith(search_str)` → `split()[-1]` |
| Check job success | `"Normal termination" in line` |
| Parse geometry | Boolean flag + header-skip counter |
| Bond lengths | Euclidean distance formula with `math.sqrt` |
| Bond angles | Dot-product / arccos formula |
| Combine everything | One-pass parser returning a structured dict |

### Where to go next

- **Multiple files**: wrap `parse_gaussian` in a loop to compare energies across
  a reaction — e.g., reactant, transition state, and product.
- **Reaction energetics**: subtract `G_total` values to compute ΔG‡ and ΔG_rxn.
- **matplotlib**: plot SCF energy vs. optimisation step or draw a reaction coordinate diagram.
- **cclib**: a dedicated Python library for parsing many quantum-chemistry codes,
  built on the same string-parsing ideas you just practiced.
